In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import json

#preprocessing
from sklearn import set_config
set_config(display = 'diagram', transform_output= "pandas")


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats


# Project config
from src.params import *
from src.utils import *
from src.preprocess.cleaning import *
from src.preprocess.features import *
from src.preprocess.pipeline import *

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Preprocessing Pipeline

This notebook builds and validates the full preprocessing pipeline, step by step.

Each section corresponds to a function in `src/preprocess/` (`cleaning.py`, `features.py`, `pipeline.py`).
The final pipeline is wrapped in `preprocessing_pipeline()` in `pipeline.py`, configured via `PreprocessConfig`.

**Input:** raw CSVs (`airqual_eda.csv`, `weather_eda.csv`) or BQ tables  
**Output:** a model-ready DataFrame with features, encoded time variables, and log-transformed target

1. get data (from local ou BQ)
2. airqual: remove negative values
3. airqual: filter sensors based on thresholds (params.py)
4. airqual: average sensors par city/date
5. merge airqual + weather → NaN apparaissent sur les jours manquants airqual
6. impute gaps ≤ 2j par interpolation linéaire (par city)
7. generate target : pm25 shift(-1) par city
8. log transform target : log(target + 1)
9. feature engineering :
   - lag features : shift(0,1,2,6,12,13,14)
   - mean(shift(12), shift(13), shift(14))
   - std(shift(0)...shift(6))
   - weather features : shift(0)
   - time features : month_sin/cos (date+1), day_of_week (date+1), is_weekend (date+1)
   - city : catégorielle native LightGBM
10. dropna → élimine gaps résiduels + début de série + dernière row


# 0. Load Data
Load raw CSVs produced by `download_data.ipynb`. These are the **pre-filter** datasets containing all cities and sensors.
The city selection and sensor filtering applied below reduce the dataset to the 6 cities validated in EDA.

## I. Cleaning
Steps applied to the raw airqual data before merge:
1. **Remove negative/zero PM2.5 values** — physically impossible readings (~1% of rows)
2. **Filter sensors** based on gap statistics and monthly coverage (thresholds in `params.py`)
3. **Select columns** — keep only `city`, `date`, `sensor_id`, `pm25_avg`
4. **Average sensors** — produce one PM2.5 value per city × date

### airqual data

In [10]:
#load airqual
df_airqual = load_data_local(filepath= "../data/raw/airqual.csv", source="airqual")
#remove neg valuues
df_airqual_no_neg = clean_neg_values(df_airqual)

✅ Loaded 17951 rows from ../data/raw/airqual.csv
⚠️  162 aberrant (negative or zero) values found (0.90%)
✅ All negative values removed — 17789 rows remaining


In [11]:
# filter sensors
df_airqual_sensor_filtered = filter_sensors(df= df_airqual_no_neg, max_gap= MAX_GAP, max_q= MAX_Q,
                                     min_bad_month_pct= MIN_BAD_MONTH_PCT, min_coverage_pct= MIN_COVERAGE_PCT)

  [gap filter] 4 sensor(s) flagged — max_gap=30d, max_q75=10.0d
  [coverage filter] 1 sensor(s) flagged — min_cov=70%, max_bad_months=20%
✅ filter_sensors: 4 sensor(s) removed, 22 remaining


In [12]:
#average sensors
df_airqual_col_selected = filter_columns(df= df_airqual_sensor_filtered, col_to_keep=["date", "city", "sensor_id", "pm25_avg"])

In [13]:
df_airqual_averaged = average_sensors(df= df_airqual_col_selected)

✅ Sensors averaged — 4278 rows (city × date)


In [14]:
df_airqual_averaged

,city,date,pm25_avg
0,Berlin,2023-05-01,13.300000
1,Berlin,2023-05-02,10.450000
2,Berlin,2023-05-03,8.456667
3,Berlin,2023-05-04,9.633333
4,Berlin,2023-05-05,10.176667
...,...,...,...
4273,Rome,2025-04-26,4.750000
4274,Rome,2025-04-27,7.500000
4275,Rome,2025-04-28,9.750000
4276,Rome,2025-04-29,8.500000


### Weather data

In [15]:
#load weather data
df_weather = load_data_local(filepath= "../data/raw/weather.csv", source= "weather")

#filter out temp avg data
df_weather_col_filtered = filter_columns(df= df_weather, col_to_remove= ["temp_avg"])

✅ Loaded 4386 rows from ../data/raw/weather.csv


## II. Merge + Impute
Left join on `date × city` — weather drives the date range, missing airqual days appear as NaN.
1-day gaps are then interpolated linearly per city (`limit=1`, `limit_area='inside'`).
Gaps > 1 day and boundary NaNs are left as-is and removed later by `dropna()`.

In [16]:
data_merged = merge_source_df(df_airqual= df_airqual_averaged, df_weather= df_weather, col_order = ["city", "date", "pm25_avg",
                                                         "temp_min", "temp_max","temp_avg",
                                                        "cloud_cover", "humidity", "precipitation",
                                                        "pressure", "wind_speed", "wind_direction"])

✅ DataFrames merged successfully — 4278/4386 days have airqual data (97.5%)


### Single-day gap imputation

In [17]:
data_imputed = single_gaps_imputer(df= data_merged)

total nan before: 108
✅ Imputing successful.1-day gaps before: 108. After: 55


## III. Target Generation
Target = PM2.5 at day J+1 (1-step ahead forecast), gene
rated by shifting `pm25_avg` by -1 per city.
Log-transformed with `log1p` to reduce skewness and stabilize variance (decision from EDA Phase III).

In [18]:
data = generate_target(data_imputed)

In [19]:
data = target_transform(data)

## IV. Feature Engineering
Features generated from the merged dataset (all lagged relative to **prediction day J**, targeting J+1):
- **PM2.5 lags**: lag_1..lag_3, lag_7 (custom) or lag_1..lag_14/21 (exhaustive)
- **Rolling mean** around lag 14: mean(lag_13, lag_14, lag_15) — captures ACF secondary peak
- **Rolling std** over 7 days: local volatility
- **Time features**: month sin/cos, day-of-week, is_weekend (for day J+1)
- **Weather features**: temp, humidity, precipitation, pressure, wind at day J
- **City**: passed as categorical to LightGBM (no encoding needed)

Two approaches available via `config.approach`: `'custom'` (selected lags) or `'all_lags_14'`/`'all_lags_21'` (exhaustive).

In [20]:
#nan for ref
data[data["pm25_avg"].isna()].index

Index([  76,  118,  293,  317,  318,  319,  320,  321,  324,  325,  326,  807,
       1779, 1780, 1781, 1782, 1783, 1786, 1787, 1788, 1935, 3241, 3242, 3243,
       3244, 3245, 3248, 3249, 3250, 3257, 3260, 3263, 3270, 3271, 3286, 3312,
       3313, 3316, 3317, 3418, 3419, 3547, 3548, 3549, 3550, 3551, 3601, 3647,
       3654, 3764, 3765, 4043, 4044, 4197, 4270],
      dtype='int64')

In [21]:
# month features
data = month_encoding(data)


In [22]:
#day features
data = day_encoding(data)

In [23]:
#average features
data = average_feature_generation(data)

In [24]:
data = std_feature_generation(data)

In [25]:
data = generate_lag_features(data, shifts= CUSTOM_SHIFTS)

## V. Full Pipeline
End-to-end run via `preprocessing_pipeline(df_airqual, df_weather, config)` from `pipeline.py`.
Verifies the complete sequence produces a clean, model-ready DataFrame.
`PreprocessConfig` controls all thresholds — see `src/params.py` for default values.

In [29]:
airqual = load_data_local(filepath= "../data/raw/airqual.csv", source="airqual")
weather = load_data_local(filepath= "../data/raw/weather.csv", source= "weather")


✅ Loaded 17951 rows from ../data/raw/airqual.csv
✅ Loaded 4386 rows from ../data/raw/weather.csv


In [30]:
data = preprocessing_pipeline(airqual_df= airqual, weather_df= weather)

⚠️  162 aberrant (negative or zero) values found (0.90%)
✅ All negative values removed — 17789 rows remaining
  [gap filter] 4 sensor(s) flagged — max_gap=30d, max_q75=10.0d
  [coverage filter] 1 sensor(s) flagged — min_cov=70%, max_bad_months=20%
✅ filter_sensors: 4 sensor(s) removed, 22 remaining
✅ Sensors averaged — 4278 rows (city × date)
✅ DataFrames merged successfully — 4278/4386 days have airqual data (97.5%)
total nan before: 108
✅ Imputing successful.1-day gaps before: 108. After: 55
✅ dropna: 367 rows removed, 4019 remaining
✅ raw data processed susscessfully!


In [31]:
save_data_local(df= data, output_path="../data/processed/df_processed.csv")

✅ Saved 4019 rows to ../data/processed/df_processed.csv
